# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining_label"].mean()
print(f"Rows: {len(df):,}   base rate: {BASE_RATE:.3f}")

print(f"""
Question: given limited review capacity (~50 pages/cycle), which of {len(df):,} pages
should a content team review first for refresh, expansion, or protection -- and why?
""")

Working directory: C:\Users\Laptop\Documents\fly


Rows: 30,000   base rate: 0.542

Question: given limited review capacity (~50 pages/cycle), which of 30,000 pages
should a content team review first for refresh, expansion, or protection -- and why?



### 1. Question

Given a portfolio of published content pages and a fixed weekly review capacity (~50 pages),
which pages should a content team review first for refresh, expansion, or protection — and why?
This mirrors the deployed paper's Introduction (see the paper for the full framing).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
print("Starter dataset: content_refresh_anonymized.csv")
print(f"  {df.shape[0]:,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")
print()
print("Warehouse data-contract exercise (separate from this model's training data):")
print("  month=2026-03: 9,841,378 rows, 55 clients, 331,437 content items (verified in w03)")
print("  finding: 88.1% of March rows show dim_content.content_updated_date AFTER report_date")
print("  finding: ga4_data_available is TRUE for only 4.2% of March rows")
print()
excluded = ["trend_direction", "trend_pct", "provider_used", "model_used", "content_id", "client_id"]
print("Excluded from modeling:", excluded)

Starter dataset: content_refresh_anonymized.csv
  30,000 rows x 45 columns, 32 clients

Warehouse data-contract exercise (separate from this model's training data):
  month=2026-03: 9,841,378 rows, 55 clients, 331,437 content items (verified in w03)
  finding: 88.1% of March rows show dim_content.content_updated_date AFTER report_date
  finding: ga4_data_available is TRUE for only 4.2% of March rows

Excluded from modeling: ['trend_direction', 'trend_pct', 'provider_used', 'model_used', 'content_id', 'client_id']


### 2. Data

Two datasets, two purposes: the starter CSV (30,000 rows, 32 clients) is what the model in this
report is actually trained and validated on; the full warehouse (queried directly in Week 3) is a
separate data-contract exercise confirming the approach is ready to scale — not the source of
this model. Both real, concrete data-safety findings from that exercise (the `dim_content`
snapshot trap, the 3-valued `ga4_data_available` flag) are carried into the paper's Data section
as scale-readiness notes, not hidden.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
import numpy as np
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count",
    "char_count", "search_volume", "competition", "cpc", "has_word_count", "has_keyword_data", "has_position"]
CATEGORICAL = ["content_type", "main_intent", "competition_level", "age_tier"]
FEATURES = NUMERIC + CATEGORICAL

leaky = set(FEATURES) & {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert not leaky, f"leakage: {leaky}"
print(f"Task: classification, deployed as ranking (precision@K). Features: {len(FEATURES)}. Leakage check: passed.")
print("Label/proxy: is_declining_label = impressions fell >20% (last 30d vs prev 30d). Base rate:", round(BASE_RATE, 3))
print("Baseline: hand-written CTR-gap-at-achievable-position rule (Week 4). Model: Logistic Regression (Week 5-7).")
print("Validation: client-grouped split throughout -- measured to matter directly (Week 6): naive split precision@50 0.90 vs grouped 0.64.")

Task: classification, deployed as ranking (precision@K). Features: 25. Leakage check: passed.
Label/proxy: is_declining_label = impressions fell >20% (last 30d vs prev 30d). Base rate: 0.542
Baseline: hand-written CTR-gap-at-achievable-position rule (Week 4). Model: Logistic Regression (Week 5-7).
Validation: client-grouped split throughout -- measured to matter directly (Week 6): naive split precision@50 0.90 vs grouped 0.64.


### 3. Methodology

Classification model (observed `is_declining_label`), deployed as a ranking tool and evaluated
with precision@K to match the real constraint of fixed review capacity. Every split in this
project is grouped by `client_id` — and Week 6 measured directly, not just assumed, why that
matters: a naive split inflates precision@50 by 26 points versus the grouped split used
throughout. Full leakage and methodology detail is in the paper's Methodology section and in
`w06_validation_audit.ipynb`.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
precision_summary = pd.DataFrame({
    "method": ["base_rate", "baseline_rule", "logistic_regression_oof"],
    "precision_at_10": [0.542, 0.30, 0.60],
    "precision_at_20": [0.542, 0.40, 0.75],
    "precision_at_50": [0.542, 0.640, 0.740],
    "precision_at_100": [0.542, 0.61, 0.80],
}).set_index("method")
print(precision_summary.to_string())
print()
print("Model clears both baseline and base rate at every K. Precision@50/@100 are the")
print("trustworthy numbers (Week 5: @10/@20 move ~10pts per single flipped item on small K).")

                         precision_at_10  precision_at_20  precision_at_50  precision_at_100
method                                                                                      
base_rate                          0.542            0.542            0.542             0.542
baseline_rule                      0.300            0.400            0.640             0.610
logistic_regression_oof            0.600            0.750            0.740             0.800

Model clears both baseline and base rate at every K. Precision@50/@100 are the
trustworthy numbers (Week 5: @10/@20 move ~10pts per single flipped item on small K).


### 4. Results (vs baseline)

Reproduced here from Week 7's out-of-fold evaluation (5-fold `GroupKFold`, full 30,000-row
coverage) and Week 4's baseline. The model clears the hand-written baseline and the base rate at
every K tested; see the paper's Results section for the charts and the error analysis (false
positives share the baseline rule's exact low-CTR-at-achievable-position blind spot).

## 5. Limitations

*What this work cannot claim.*

In [5]:
print("Limitations (see paper for full discussion):")
for item in [
    "Label is a proxy (measured decline), not a confirmed content problem",
    "Client concentration persists in both baseline and model queues (Week 4 + Week 7)",
    "Single point-in-time snapshot -- not a live feed, not yet the 79M-row warehouse",
    "Observational only -- no causal claims, no claim about search engine behavior",
    "One grouped split compared models; a repeated grouped CV would strengthen that comparison",
]:
    print(" -", item)

Limitations (see paper for full discussion):
 - Label is a proxy (measured decline), not a confirmed content problem
 - Client concentration persists in both baseline and model queues (Week 4 + Week 7)
 - Single point-in-time snapshot -- not a live feed, not yet the 79M-row warehouse
 - Observational only -- no causal claims, no claim about search engine behavior
 - One grouped split compared models; a repeated grouped CV would strengthen that comparison


### 5. Limitations

The single most important repeated finding across this project is the client-concentration
pattern — found independently in the Week 4 hand-rule and the Week 7 validated model, meaning
it's a property of the data's client distribution, not of either scoring method.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
import json
with open("work/outputs/w07_metrics.json") as f:
    w07_metrics = json.load(f)
print("Suggested action mix (from Week 7's committed metrics):")
for action, count in w07_metrics["suggested_action_mix"].items():
    print(f"  {action}: {count:,}")
print()
print("Reason code -> action: ctr_gap_at_achievable_position -> review_ctr_then_refresh")
print("                        stale_visible_page -> refresh")
print("                        thin_visible_page -> expand_and_refresh")
print("                        (all gated: low confidence always routes to monitor)")

Suggested action mix (from Week 7's committed metrics):
  monitor: 18,875
  refresh: 5,677
  review_ctr_then_refresh: 5,400
  expand_and_refresh: 48

Reason code -> action: ctr_gap_at_achievable_position -> review_ctr_then_refresh
                        stale_visible_page -> refresh
                        thin_visible_page -> expand_and_refresh
                        (all gated: low confidence always routes to monitor)


### 6. Ranked recommendations

The action playbook from Week 7, condensed: reason codes map to suggested actions, gated by the
model's own confidence — a page never gets a stronger-than-`monitor` action if the model isn't
confident about it, regardless of which archetype it matches. Full no-go list and monitoring
triggers are in the paper's Ranked Recommendations section.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import shutil, os
os.makedirs("docs/assets", exist_ok=True)
# the paper's charts are generated by scripts/notebooks already run this week;
# this cell confirms they're present and lists them as the artifacts the
# deployed page embeds.
for f in sorted(os.listdir("docs/assets")):
    print("docs/assets/", f, f"({os.path.getsize(os.path.join('docs/assets', f)):,} bytes)")

docs/assets/ action_mix.svg (29,954 bytes)
docs/assets/ model_vs_baseline.svg (42,915 bytes)
docs/assets/ split_leakage.svg (40,983 bytes)


### 7. Artifacts the paper embeds

`docs/assets/` holds the three charts the deployed paper (`docs/index.html`) references:
`model_vs_baseline.svg` (Results), `split_leakage.svg` (Methodology, the honest-split proof), and
`action_mix.svg` (Ranked recommendations, reused from Week 7's committed figure).

## ML-12 — Demo outline and shareable cuts

### 5-minute demo outline
1. **Question** (30s): out of thousands of content pages, which ~50 should a reviewer touch
   this cycle?
2. **Method** (90s): a hand-written baseline vs. a client-grouped, leakage-audited Logistic
   Regression model — same data, same metric.
3. **One chart** (90s): `docs/assets/split_leakage.svg` — same model, same data, only the split
   changed, and precision@50 moved 26 points. This is the single most convincing image in the
   project because it's a measurement, not an assumption.
4. **One honest result** (60s): 0.740 precision@50 vs. a 0.640 baseline and a 0.542 base rate —
   real lift, not dramatic, and I can say exactly why it isn't dramatic (small-K noise, proxy
   label, client concentration).
5. **One recommendation** (30s): ship the ranked queue as a reviewer aid with mandatory review on
   medium-confidence rows — never as an autonomous action.

### Social post (methodology-focused)
> Spent 8 weeks building a content-refresh priority model on FlyRank's ML internship dataset.
> The most useful finding wasn't the model — it was proving *why* validation design matters: the
> exact same model, same data, scored 0.90 precision@50 on a naive split and 0.64 on a
> client-grouped one. A 26-point swing from one design choice. Full writeup + reproducible
> notebooks: [repo link].

### Employer-facing summary (3 sentences)
I built and validated a content-prioritization model on FlyRank's 30,000-page ML internship
dataset, comparing a transparent baseline rule against a Logistic Regression model under a
client-grouped holdout. The validated model reached 0.740 precision on its top 50 recommendations
(vs. a 0.542 base rate), and I directly measured — rather than assumed — that a naive split would
have overstated that same model by 26 points. The output is a ranked, reason-coded action queue
with explicit human-review rules, deployed as a public research paper with fully reproducible
notebooks.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
